In [ ]:
from pathlib import Path
import json
import time

from faster_whisper import WhisperModel
from tqdm.auto import tqdm

In [ ]:
ROOT_DIR = Path.cwd()

AUDIO_DIR = ROOT_DIR / "DAKE_output" / "extracted_audios"

OUTPUT_DIR = ROOT_DIR / "DAKE_output" / "extracted_subtitles"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

audio_paths = sorted(AUDIO_DIR.glob("*.mp3"))

print(f"Found {len(audio_paths)} audio files")

In [ ]:
MODEL_NAME = "large-v3-turbo"

DEVICE = "cpu"

COMPUTE_TYPE = "int8"

model = WhisperModel(
    MODEL_NAME,
    device=DEVICE,
    compute_type=COMPUTE_TYPE,
)

print("Model loaded.")

In [ ]:
def transcribe_audio(audio_path: Path):

    output_path = OUTPUT_DIR / f"{audio_path.stem}.json"

    if output_path.exists():
        return

    segments, info = model.transcribe(
        str(audio_path),
        language="vi",
        vad_filter=True,
    )

    subtitles = []

    for seg in segments:

        subtitles.append(
            {
                "start": round(seg.start, 3),
                "end": round(seg.end, 3),
                "text": seg.text.strip(),
            }
        )

    with open(output_path, "w", encoding="utf-8") as f:

        json.dump(
            subtitles,
            f,
            ensure_ascii=False,
            indent=2,
        )

In [ ]:
start = time.time()

for audio_path in tqdm(audio_paths):

    transcribe_audio(audio_path)

print(f"Done in {(time.time()-start)/60:.2f} minutes")